# Tests using OpenCV


Importamos las librerias necesarios (incluyendo la libreria openCV) y verificamos que version se tiene instalada (para confirmar su instalacion):

In [2]:
import cv2 as cv
import numpy as np
import sys
import pymupdf
from pathlib import Path

print(cv.__version__)

4.13.0


## Basic Operation on images

---

Here we practice basic operations that can be done on images that will be later relied on when performing the script to parse the finantial data from the pdf to a pandas dataframe.

### Accessing and modifying pixel values

We start by loading the colored image associated to the finantial statements of one year:


In [3]:
image_path = Path("../OUTPUTS/page_0_cut.png")

img = cv.imread(str(image_path))
img_gray = cv.imread(str(image_path), cv.IMREAD_GRAYSCALE)

We can access a pixel value by its row and column coordinates. For a colored image (BGR by default), it returns an array of Blue, Green, Red values.

In [4]:
if img is not None:
    pixel = img[200,200]
    print(pixel)

if img_gray is not None:
    pixel_gray = img_gray[200, 200]
    print(pixel_gray)

[255 255 255]
255


We can modify the pixel values in the same way:

In [5]:
# img[100,100] = [255,255,255]
# print( img[100,100] )

### Access image properties

We can inspect images read using imread to get their bassic properties; aspects like dimensions, color channels, data types and pixel values can be inspected. We start by inspecting the properties of the image relevant to its **size**. For this we use the method `.shape()`. 

First we inspect the properties of the colored image:


In [6]:
if img is not None:
    height, width, channels = img.shape
    print(f"Image Height: {height} pixels")
    print(f"Image Width: {width} pixels")
    print(f"Number of Color Channels: {channels}")

    # Calculate total number of pixels
    total_pixels = height * width
    print(f"Total Pixels: {total_pixels}")
else:
    print(f"Error: Could not read image file at {image_path}")

Image Height: 5500 pixels
Image Width: 4250 pixels
Number of Color Channels: 3
Total Pixels: 23375000


If the image is in **grayscale**, when loading the image, the shape will only have two values: `(height, width)`. There is no third dimension for color channels because each pixel has only one intensity value (by default from 0 to 256 inclusive). As such, the previous code needs to be slightly modified:

In [7]:
# This snippet works for both color and grayscale images, but we need to check dimensions first to avoid unpacking errors.

if img_gray is not None:
    # Check dimensions - will raise an error if you try to unpack 3 values
    if len(img_gray.shape) == 2:
        height, width = img_gray.shape
        channels = 1 # Grayscale has 1 channel
        print(f"(Grayscale) Height: {height}, Width: {width}, Channels: {channels}")
    # Handle potential color image loaded unexpectedly (shouldn't happen with flag 0)
    elif len(img_gray.shape) == 3:
        height, width, channels = img_gray.shape
        print(f"(Color?) Height: {height}, Width: {width}, Channels: {channels}")
else:
     print(f"Error: Could not read image file at {image_path}")

(Grayscale) Height: 5500, Width: 4250, Channels: 1


We can also check the data type. Digital images strore each pixel value using a specific numerical data type. This data type determines the range of possible values for each pixel componen (3 components when working with BGR). OpenCV uses by default `uint8`, which means that each color component `(B, G, R)` is represented by an integer between 0 and 256.

In [8]:
# Make sure the image loaded before proceeding
if img is not None:
    # Get the data type of the image array
    data_type = img.dtype
    print(f"Image Data Type: {data_type}")
else:
    print(f"Error: Could not read image file at {image_path}")

Image Data Type: uint8


We can also access the values of specific pixels using numpy indexing. For this we use the fact that images use a coordinate system where the origin (0,0) is the top left-most pixel. Here we se an example of accessing specific pixel's values:

In [9]:
if img is not None:
    # Define coordinates (y, x) - remember y is row, x is column
    px_y, px_x = 2000, 1000 # Example coordinates

    # Check if coordinates are within image bounds
    height, width, _ = img.shape
    if 0 <= px_y < height and 0 <= px_x < width:
        # Access the pixel value at (y, x)
        pixel_value = img[px_y, px_x]
        print(f"Pixel value at ({px_y}, {px_x}): {pixel_value}")

        # For a color image (BGR), pixel_value is an array [Blue, Green, Red]
        blue = pixel_value[0]
        green = pixel_value[1]
        red = pixel_value[2]
        print(f"  Blue: {blue}, Green: {green}, Red: {red}")
    else:
        print(f"Coordinates ({px_y}, {px_x}) are out of image bounds ({height}x{width}).")
else:
    print(f"Error: Could not read image file at {image_path}")

Pixel value at (2000, 1000): [255 255 255]
  Blue: 255, Green: 255, Red: 255


### Set a region of interest

Sometimes we will need to play with a certain regions of interest (ROI). These regions can be obtained by using Numpy indexing. As an example, in the sequel we copy a rectangle region in the colored image and paste it in another regions within the same image; we then store the resulting image.


In [10]:
output_image_path = Path("../OUTPUTS/page_0_modified.png")

if img is not None:

    rectangle = img[1000:2000, 1000:2000] # This will be a 1000x1000 pixel region
    img[3000:4000, 3000:4000] = rectangle # Copy the rectangle to a new location

    output_image_path.unlink(missing_ok=True) # Remove existing file if it exists
    cv.imwrite(str(output_image_path), img)
else:
    print(f"Error: Could not read image file at {image_path}")


La imagen resultante es:

![imagen](../OUTPUTS/page_0_modified.png)

## Applying a threshold to a grayscaled image

---

When working with an image, it is sometimes easy to identify information that is not relevant for our purposes based on its intensity (intensity of the pixel values). For that, an inteligent first step can be to apply a threshold to "clean" the image of non-relevant information.

We do this using the `.threshold()` function, which has 4 parameters:

1. str: Input array (for our purposes, a grayscaled image).
2. thresh: Threshold value.
3. maxval: Maximum value to use with the THRESH_BINARY and THRESH_BINARY_INV thresholding types. 
4. type: thresholding type.

It returns the computed threshold value (if we use methods that automatically calculate a threshold value), and the output array (i.e. the resulting image).

In the sequel, we show a snippet that reads the original image, applying a binary threshold for a specific threshold value, and saves the resulting image in the OUTPUTS folder.

In [11]:
pre_threshold_image = cv.imread(str(image_path), cv.IMREAD_GRAYSCALE)

if pre_threshold_image is not None:
    _, threshold_image = cv.threshold(pre_threshold_image, 10, 255, cv.THRESH_BINARY)
    print(f"The .threshold function also returns the thtreshold value, even if it was given as a parameter\nSaid value is: {_}")
    threshold_image_path = Path("../OUTPUTS/page_0_threshold.png")
    threshold_image_path.unlink(missing_ok=True)
    cv.imwrite(str(threshold_image_path), threshold_image)
else:
    print(f" Error reading the file at {image_path}")

The .threshold function also returns the thtreshold value, even if it was given as a parameter
Said value is: 10.0


The previous snippet essentially does the following:

1. Reads the image in grayscale mode.
2. Applies a binary threshold in the grayscaled image, where any pixel that has a pixel value below 10, is immediatelly changed to a pixel value of 255 (i.e. white).
3. Stores the reuslting image (shich is shown in the sequel)

![image](../OUTPUTS/page_0_threshold.png)

> Notice that a lot of information regarding the finantial statements is lost. The purpose of applying threshold is to clearn the image and facilitate shape recognition (at least in our case).

## Detecting edges in images

---

Edges in images represent intense changes in intensity. The Canny edge detector is a popular and effective algorithm for identifying these edges. We will use an implementation of said algorithm in OpenCV.

> Note that it is common practice to work with grayscale images for edge detection, as color information isn't needed to find intensity changes.

With the grayscale image loaded, applying the Canny edge detector in OpenCV is straightforward using the .Canny() function. The main parameters we need to provide are:

- image: The input grayscale image
- threshold 1: The lower threshold for the hysteresis procedure
- threshold 2: The upper threshold for the hysteresis procedure

To understand what these thresholds do, the edges with intensity above `threshold 2` are definite edges; edges below `threshold 1` are discarted; edges between the two thresholds are kept if they are connected to a definite edge.

In [12]:
threshold_image = cv.imread(str(threshold_image_path), cv.IMREAD_GRAYSCALE)

if threshold_image is not None:
    # Apply the Canny edge detector
    # Using threshold1=100 and threshold2=200 as starting values
    edges = cv.Canny(threshold_image, 1000, 1100)

    # The 'edges' variable now holds the edge map image
    # It's a binary image where white pixels represent detected edges
    edges_detection_image = Path(f"../OUTPUTS/page_0_edges.png")
    edges_detection_image.unlink(missing_ok=True)
    cv.imwrite(str(edges_detection_image), edges)
    height, width = edges.shape
    print(f"The edge map image has dimensions: {height}x{width} pixels")
else:
    print(f"Error: Could not read image file at {threshold_image_path}")


The edge map image has dimensions: 5500x4250 pixels


The previous code snippet returns the following image:

![image](../OUTPUTS/page_0_edges.png)

> This image will be very usefull when performing countour detection, which will be highly usefull when partitioning the original image in subimages to then apply OCR in said subsections.

## Countour detection

---

**Countours** can be explanied as curves joining all the continuous points along a boudary, having the same color or intensity. The contours are useful for shape analysis and object recognition.

In the sequel we place a simple code snippet to find the countours of the previous edges image:

In [14]:
edges_image = cv.imread(str(edges_detection_image), cv.IMREAD_GRAYSCALE)

if edges_image is not None:
    edges_image_countours, countours = cv.findContours(edges_image, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    image_countors_path = Path("../OUTPUTS/page_0_edges_contours.png")
    image_countors_path.unlink(missing_ok=True)
    cv.imwrite(str(image_countors_path), edges_image)
    print(countours)
else:
    print(f"Error reading edges image at {edges_detection_image}")

[[[   1   -1   -1   -1]
  [   2    0   -1   -1]
  [   3    1   -1   -1]
  ...
  [8658 8656   -1   -1]
  [8659 8657   -1   -1]
  [  -1 8658   -1   -1]]]


In [ ]:
x_diff = []
for countour in countours[0]:
    x_diff.append(countour[0]

8660
